In [1]:
import os
import sys
from ipywidgets import interact

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)
 
from syn_project.utils_train import *
from syn_project.utils_notebook import *

%matplotlib widget

In [2]:
condition = "shape"
data = "biased_00"
switch_epoch = 10000
checkpoint_epoch = 0

n_samples = 32
show_results_fusion = True
fusion_attr_weight = 1.0

project_name = "syn"
experiment_name = get_experiment_name(condition, data, switch_epoch)


training_params = get_training_params(project_name, experiment_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

global_workspace = get_global_workspace(project_name, experiment_name, epoch=checkpoint_epoch)
data_module = get_data_module(project_name,  experiment_name)
train_samples = get_data_samples(data_module, n_samples)
data_translated = get_data_translated(global_workspace, train_samples, n_samples, fusion_attr_weight, show_results_fusion)

/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/lightning/fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


Loading model from checkpoint: /home/lucas/gwsyn/checkpoints/syn/shape_biased_00_switch_10000/checkpoints/last.ckpt
Loaded default weights from /home/lucas/gwsyn/checkpoints/syn/shape_biased_00_switch_10000/checkpoints/last.ckpt


In [3]:
unimodal_latents = global_workspace.encode_domains(train_samples)
gw_latents = global_workspace.encode(unimodal_latents)
gw_latent = gw_latents[frozenset({'attr', 'v_latents'})]
gw_latents_decoded = global_workspace.decode(gw_latent, ["v_latents", "attr"])

In [4]:
visual_module = cast(VisualLatentDomainModule, global_workspace.domain_mods["v_latents"])
images = visual_module.decode_images(gw_latents_decoded["v_latents"]["v_latents"]).detach().cpu()


In [5]:
from typing import cast
from shimmer import GlobalWorkspace2Domains
from shimmer_ssd.modules.domains.visual import VisualLatentDomainModule


def get_decoded_image_from_gw_latent(gw_latent_array: np.ndarray, global_workspace:GlobalWorkspace2Domains, device: torch.device):
    gw_latent_tensor = {'attr': torch.Tensor([gw_latent_array]).to(device)}
    gw_latents_decoded = global_workspace.decode(gw_latent_tensor, ["v_latents", "attr"])
    
    visual_module = cast(VisualLatentDomainModule, global_workspace.domain_mods["v_latents"])
    
    images = visual_module.decode_images(gw_latents_decoded['attr']["v_latents"]).detach().cpu()

    return images[0].permute(1, 2, 0).detach().cpu().numpy()

In [6]:
# On définit les réglages des 12 curseurs (-1 à 1, pas de 0.1)
sliders = {f'v{i+1}': (-1.0, 1.0, 0.1) for i in range(12)}

@interact(
    **sliders
)
def play_with_gw(**values):
    latent_vector = np.array([values[k] for k in sorted(values.keys())], dtype=np.float32)
    img = get_decoded_image_from_gw_latent(latent_vector, global_workspace, device)
    plt.imshow(img)
    plt.axis('off') # Souvent plus joli pour des images générées
    plt.show()


interactive(children=(FloatSlider(value=0.0, description='v1', max=1.0, min=-1.0), FloatSlider(value=0.0, desc…